In [24]:
# Import required libraries and create a Spark Session with Delta Lake support
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("Delta Lake MERGE Assignment")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 3.5.6


In [25]:
# Load the customer dataset (Sample Superstore CSV) into a Spark DataFrame
df = spark.read.csv(
    "../data/Sample - Superstore.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [26]:
# Display the schema to understand the dataset structure and data types
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [27]:
# Count the total number of rows and columns in the dataset
print("Total Rows :", df.count())
print("Total Columns :", len(df.columns))

Total Rows : 9994
Total Columns : 21


In [28]:
# Check for null values in each column
from pyspark.sql.functions import col, count, when

null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])


null_counts.show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [29]:
# Replace null values in string columns with 'Unknown'
from pyspark.sql.types import StringType

string_columns = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
]

df_clean = df.fillna("Unknown", subset=string_columns)

In [30]:
# Remove duplicate records from the dataset
before = df_clean.count()

df_clean = df_clean.dropDuplicates()

after = df_clean.count()

print(f"Rows before removing duplicates : {before}")
print(f"Rows after removing duplicates  : {after}")
print(f"Duplicates removed              : {before-after}")

Rows before removing duplicates : 9994
Rows after removing duplicates  : 9994
Duplicates removed              : 0


In [31]:
# Display the cleaned dataset
df_clean.show(5)

+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|Row ID|      Order ID|Order Date|Ship Date|     Ship Mode|Customer ID|   Customer Name|    Segment|      Country|         City|       State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount| Profit|
+------+--------------+----------+---------+--------------+-----------+----------------+-----------+-------------+-------------+------------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+-------+
|   188|CA-2016-157000| 7/16/2016|7/22/2016|Standard Class|   AM-10360|  Alice McCarthy|  Corporate|United States|Grand Prairie|       Texas|      75051|Central|OFF-ST-10001328|Office Supplies|     Storage|Personal

In [32]:
# Rename columns to remove spaces and special characters for Delta Lake compatibility
df_clean = df_clean.toDF(*[
    c.replace(" ", "_")
     .replace("-", "_")
     .replace("/", "_")
    for c in df_clean.columns
])

print(df_clean.columns)

['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [33]:
# Save the cleaned dataset as a Delta table
import shutil
import os

delta_path = "../data/delta_customer"

if os.path.exists(delta_path):
    shutil.rmtree(delta_path)

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

print("Delta table created successfully!")

Delta table created successfully!


In [34]:
# Read the Delta table for further processing
delta_df = spark.read.format("delta").load(delta_path)

delta_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+-------------+------------+-----------+------+---------------+----------+------------+--------------------+--------+--------+--------+-------+
|Row_ID|      Order_ID|Order_Date| Ship_Date|     Ship_Mode|Customer_ID|  Customer_Name|  Segment|      Country|         City|       State|Postal_Code|Region|     Product_ID|  Category|Sub_Category|        Product_Name|   Sales|Quantity|Discount| Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+-------------+------------+-----------+------+---------------+----------+------------+--------------------+--------+--------+--------+-------+
|   689|CA-2017-161480|12/25/2017|12/29/2017|Standard Class|   RA-19285|   Ralph Arnett| Consumer|United States|New York City|    New York|      10035|  East|FUR-BO-10004015| Furniture|   Bookcases|Bush Andora Bookc...| 191.984|       

In [35]:
# Create a sample incremental dataset containing updated and new records
incremental_data = [
    (1, "CA-2016-152156", "Updated Customer", "Consumer", "United States", "Kentucky"),
    (9995, "NEW-001", "New Customer 1", "Corporate", "United States", "Texas"),
    (9996, "NEW-002", "New Customer 2", "Home Office", "United States", "California")
]

incremental_df = spark.createDataFrame(
    incremental_data,
    [
        "Row_ID",
        "Order_ID",
        "Customer_Name",
        "Segment",
        "Country",
        "State"
    ]
)

incremental_df.show()

+------+--------------+----------------+-----------+-------------+----------+
|Row_ID|      Order_ID|   Customer_Name|    Segment|      Country|     State|
+------+--------------+----------------+-----------+-------------+----------+
|     1|CA-2016-152156|Updated Customer|   Consumer|United States|  Kentucky|
|  9995|       NEW-001|  New Customer 1|  Corporate|United States|     Texas|
|  9996|       NEW-002|  New Customer 2|Home Office|United States|California|
+------+--------------+----------------+-----------+-------------+----------+



In [36]:
# Perform Delta Lake MERGE to update existing records and insert new records
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, delta_path)

(
    delta_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.Row_ID = source.Row_ID"
    )
    .whenMatchedUpdate(set={
        "Customer_Name": "source.Customer_Name",
        "Segment": "source.Segment",
        "Country": "source.Country",
        "State": "source.State"
    })
    .whenNotMatchedInsert(values={
        "Row_ID": "source.Row_ID",
        "Order_ID": "source.Order_ID",
        "Customer_Name": "source.Customer_Name",
        "Segment": "source.Segment",
        "Country": "source.Country",
        "State": "source.State"
    })
    .execute()
)

print("MERGE completed successfully!")

MERGE completed successfully!


In [37]:
# Validate the merge by checking the final row count and duplicate Row_ID values
final_df = spark.read.format("delta").load(delta_path)

print("Final Row Count:", final_df.count())

print("Duplicate Row_IDs:")
final_df.groupBy("Row_ID").count().filter("count > 1").show()

Final Row Count: 9996
Duplicate Row_IDs:
+------+-----+
|Row_ID|count|
+------+-----+
+------+-----+



In [38]:
# Verify that the existing record has been updated successfully
from pyspark.sql.functions import col

final_df.filter(col("Row_ID") == 1).show(truncate=False)

+------+--------------+----------+----------+------------+-----------+----------------+--------+-------------+---------+--------+-----------+------+---------------+---------+------------+---------------------------------+------+--------+--------+-------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode   |Customer_ID|Customer_Name   |Segment |Country      |City     |State   |Postal_Code|Region|Product_ID     |Category |Sub_Category|Product_Name                     |Sales |Quantity|Discount|Profit |
+------+--------------+----------+----------+------------+-----------+----------------+--------+-------------+---------+--------+-----------+------+---------------+---------+------------+---------------------------------+------+--------+--------+-------+
|1     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class|CG-12520   |Updated Customer|Consumer|United States|Henderson|Kentucky|42420      |South |FUR-BO-10001798|Furniture|Bookcases   |Bush Somerset Collection Bookcase|261.96|2      

In [39]:
# Verify that the new records have been inserted successfully
final_df.filter(final_df.Row_ID >= 9995).show(truncate=False)

+------+--------+----------+---------+---------+-----------+--------------+-----------+-------------+----+----------+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row_ID|Order_ID|Order_Date|Ship_Date|Ship_Mode|Customer_ID|Customer_Name |Segment    |Country      |City|State     |Postal_Code|Region|Product_ID|Category|Sub_Category|Product_Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+--------------+-----------+-------------+----+----------+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|9995  |NEW-001 |NULL      |NULL     |NULL     |NULL       |New Customer 1|Corporate  |United States|NULL|Texas     |NULL       |NULL  |NULL      |NULL    |NULL        |NULL        |NULL |NULL    |NULL    |NULL  |
|9996  |NEW-002 |NULL      |NULL     |NULL     |NULL       |New Customer 2|Home Office|United States|NULL|California|NULL       |NULL  |NULL    

In [40]:
# Display the final Delta table after the MERGE operation
final_df.show(20, truncate=False)

+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+----------------------------------------------------------+--------+--------+--------+--------+
|Row_ID|Order_ID      |Order_Date|Ship_Date |Ship_Mode     |Customer_ID|Customer_Name   |Segment    |Country      |City         |State         |Postal_Code|Region |Product_ID     |Category       |Sub_Category|Product_Name                                              |Sales   |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+----------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+----------------------------------------------------------+--------+--------+--------+--------+
|689   |CA-2017-161480|12/25/2017|12/29/2017|Standard Class|RA-19285   |Ralph Arnett 

In [41]:
from pyspark.sql.functions import col

final_df.filter(col("Row_ID").isin(9995, 9996)).show(truncate=False)

+------+--------+----------+---------+---------+-----------+--------------+-----------+-------------+----+----------+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row_ID|Order_ID|Order_Date|Ship_Date|Ship_Mode|Customer_ID|Customer_Name |Segment    |Country      |City|State     |Postal_Code|Region|Product_ID|Category|Sub_Category|Product_Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+--------------+-----------+-------------+----+----------+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|9995  |NEW-001 |NULL      |NULL     |NULL     |NULL       |New Customer 1|Corporate  |United States|NULL|Texas     |NULL       |NULL  |NULL      |NULL    |NULL        |NULL        |NULL |NULL    |NULL    |NULL  |
|9996  |NEW-002 |NULL      |NULL     |NULL     |NULL       |New Customer 2|Home Office|United States|NULL|California|NULL       |NULL  |NULL    

In [45]:
# Save the cleaned master dataset as a CSV file

df_clean.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("../data/customer_master")

In [46]:
# Save the incremental dataset as a CSV file

incremental_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("../data/customer_incremental")